## STW7048CEM: Computing Individual Research Project

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/sage-shift/data
!curl -sS -o /content/r36.crt http://crt.sectigo.com/SectigoPublicServerAuthenticationCADVR36.crt
!openssl x509 -inform DER -in /content/r36.crt -out /content/r36.pem && cat /etc/ssl/certs/ca-certificates.crt /content/r36.pem > /content/ca-simula.pem
!wget -c -q --show-progress --ca-certificate=/content/ca-simula.pem -O /content/hk.zip https://datasets.simula.no/downloads/hyper-kvasir/hyper-kvasir-labeled-images.zip
!ls -l /content/hk.zip && sha256sum /content/hk.zip && unzip -tq /content/hk.zip
!cp /content/hk.zip /content/drive/MyDrive/sage-shift/data/hyper-kvasir-labeled-images.zip

In [ ]:
from google.colab import drive, userdata
import base64, os, subprocess

drive.mount("/content/drive")
REPO = "https://github.com/tek-raj-bhatt-250069/sage-europe-nepal-shift.git"
auth = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()

def git(*args):
    done = subprocess.run(["git", "-c", f"http.extraHeader=Authorization: Basic {auth}", *args],
                          capture_output=True, text=True)
    if done.returncode:
        raise SystemExit(f"git failed: {done.stderr.strip()}")

if os.path.isdir("/content/sage-shift/.git"):
    git("-C", "/content/sage-shift", "pull", "--quiet")
else:
    git("clone", "--quiet", REPO, "/content/sage-shift")
del auth
%cd /content/sage-shift
!git log -1 --format="%h %s" && nvidia-smi -L

In [ ]:
!pip install -q -r requirements.txt > /content/pip_install.log 2>&1 && pip install -q --no-deps -e . >> /content/pip_install.log 2>&1; echo "install exit code $?"
!python -c "import numpy, pandas, PIL, scipy, sklearn, sage_shift; print('numpy', numpy.__version__, '| pandas', pandas.__version__, '| pillow', PIL.__version__, '| scipy', scipy.__version__, '| scikit-learn', sklearn.__version__)"

In [ ]:
import os
from google.colab import userdata
os.environ["SYNAPSE_AUTH_TOKEN"] = userdata.get("SYNAPSE_AUTH_TOKEN")
!python scripts/00_download_sage.py

In [ ]:
%%bash
mkdir -p external/hyperkvasir/official_splits
cp /content/drive/MyDrive/sage-shift/data/hyper-kvasir-labeled-images.zip /content/
unzip -q -o /content/hyper-kvasir-labeled-images.zip -d external/hyperkvasir/
ls external/hyperkvasir
cd external/hyperkvasir/official_splits
BASE=https://raw.githubusercontent.com/simula/hyper-kvasir/21cc366e78c0cb4e180a26a0e441d6c0d5171da9/official_splits
wget -q -O 2_fold_split.csv $BASE/2_fold_split.csv
wget -q -O 5_fold_split.csv $BASE/5_fold_split.csv
sha256sum -c <<'SUMS'
e9bc264da8dd4ce5ebd6654080eb9ec8a00fd28c3d03a75a2c80b1eb5a096c06  2_fold_split.csv
0230b465bdf73c23e949c74e1d4d68c4905a758195ea6b69b10f32c71a3be426  5_fold_split.csv
SUMS

In [ ]:
!python scripts/03_build_manifests.py --out-dir /content/drive/MyDrive/sage-shift/manifests

In [ ]:
!python scripts/06_extract_features.py --variants raw --workers 4 --out-dir /content/drive/MyDrive/sage-shift/features

In [ ]:
!python scripts/05_check_fov.py --out-dir /content/preprocess
!mkdir -p /content/drive/MyDrive/sage-shift/preprocess
!cp /content/preprocess/*.json /content/preprocess/*.csv /content/preprocess/sheets/hk_* /content/drive/MyDrive/sage-shift/preprocess/

In [ ]:
!python scripts/06_extract_features.py --variants fov colour fov_colour --workers 2 \
    --reference /content/drive/MyDrive/sage-shift/preprocess/preprocess_reference.json \
    --out-dir /content/drive/MyDrive/sage-shift/features

In [ ]:
!python scripts/08_paper_recipe.py --recipe faithful --workers 2 --cache-dir /content/cache \
    --reference /content/drive/MyDrive/sage-shift/preprocess/preprocess_reference.json \
    --out-dir /content/drive/MyDrive/sage-shift/paper_recipe

In [ ]:
!python scripts/08_paper_recipe.py --recipe adjusted --workers 2 --cache-dir /content/cache \
    --reference /content/drive/MyDrive/sage-shift/preprocess/preprocess_reference.json \
    --out-dir /content/drive/MyDrive/sage-shift/paper_recipe

In [ ]:
for seed in (1, 2):
    !python scripts/08_paper_recipe.py --recipe faithful --seed {seed} --epochs 40 --workers 2 --cache-dir /content/cache \
        --reference /content/drive/MyDrive/sage-shift/preprocess/preprocess_reference.json \
        --out-dir /content/drive/MyDrive/sage-shift/paper_recipe
    !python scripts/08_paper_recipe.py --recipe adjusted --seed {seed} --workers 2 --cache-dir /content/cache \
        --reference /content/drive/MyDrive/sage-shift/preprocess/preprocess_reference.json \
        --out-dir /content/drive/MyDrive/sage-shift/paper_recipe

In [ ]:
COMMIT = "a1bbef1d40a49f42c2f07417fb67adb638a2bbc1"
from google.colab import drive, userdata
import base64, os, re, subprocess

assert re.fullmatch(r"[0-9a-f]{7,40}", COMMIT), "COMMIT must be the pushed commit hash"
drive.mount("/content/drive")
REPO = "https://github.com/tek-raj-bhatt-250069/sage-europe-nepal-shift.git"
auth = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()

def git(*args):  # the token travels in a one-off header: never in .git/config or in an error message
    done = subprocess.run(["git", "-c", f"http.extraHeader=Authorization: Basic {auth}", *args],
                          capture_output=True, text=True)
    if done.returncode:
        raise SystemExit(f"git failed: {done.stderr.strip()}")

if os.path.isdir("/content/sage-shift/.git"):
    git("-C", "/content/sage-shift", "fetch", "--quiet", "origin")
else:
    git("clone", "--quiet", REPO, "/content/sage-shift")
git("-C", "/content/sage-shift", "checkout", "--quiet", COMMIT)
del auth
%cd /content/sage-shift
!git log -1 --format="%h %s" && nvidia-smi -L
!pip install -q -r requirements.txt > /content/pip_install.log 2>&1 && pip install -q --no-deps -e . >> /content/pip_install.log 2>&1; echo "install exit code $?"
!python -c "import numpy, pandas, PIL, scipy, sklearn, sage_shift; print('numpy', numpy.__version__, '| pandas', pandas.__version__, '| pillow', PIL.__version__, '| scipy', scipy.__version__, '| scikit-learn', sklearn.__version__)"

In [ ]:
import json, os
PRE = "/content/drive/MyDrive/sage-shift/preprocess"
if os.path.isfile(f"{PRE}/colour_global_sage.json"):
    print("using the uploaded colour_global_sage.json")
else:
    !python scripts/16_colour_global_stats.py --quality {PRE}/sage_fov_quality.csv --reference {PRE}/preprocess_reference.json --out {PRE}/colour_global_sage.json
stats = json.load(open(f"{PRE}/colour_global_sage.json"))
print("frames", stats["images"], "| Lab mean", stats["mean"], "| SD", stats["std"], "| table", stats["quality_table_sha256"][:12])

In [ ]:
PRE = "/content/drive/MyDrive/sage-shift/preprocess"
FEAT = "/content/drive/MyDrive/sage-shift/features"
!python scripts/06_extract_features.py --datasets sage --workers 2 \
    --variants masked masked_grey letterbox letterbox_grey fov_grey colour_global fov_colour_global \
    --reference {PRE}/preprocess_reference.json --colour-global {PRE}/colour_global_sage.json --out-dir {FEAT}
!python scripts/06_extract_features.py --datasets hk --workers 2 --variants sageify sageify_grey \
    --reference {PRE}/preprocess_reference.json --out-dir {FEAT}

In [ ]:
for seed in (0, 1, 2):
    !python scripts/08_paper_recipe.py --recipe adjusted --seed {seed} --keep-model --workers 2 --cache-dir /content/cache \
        --reference /content/drive/MyDrive/sage-shift/preprocess/preprocess_reference.json \
        --out-dir /content/drive/MyDrive/sage-shift/paper_recipe_keep

In [ ]:
!python scripts/14_geometry_controls.py --runs-dir /content/drive/MyDrive/sage-shift/paper_recipe_keep \
    --reference /content/drive/MyDrive/sage-shift/preprocess/preprocess_reference.json \
    --cache-dir /content/cache --workers 2

In [ ]:
import os
BASE = "/content/drive/MyDrive/sage-shift"
BACKBONES = ("resnet50_imagenet", "resnet50_dino_imagenet", "dinov2_vitb14")
SAGE_INPUTS = ("masked", "masked_grey", "letterbox", "letterbox_grey", "fov_grey", "colour_global", "fov_colour_global")
files = [f"{BASE}/features/features_run.json"]
files += [f"{BASE}/features/sage/{b}/{v}.npz" for b in BACKBONES for v in SAGE_INPUTS]
files += [f"{BASE}/features/hk/{b}/{v}.npz" for b in BACKBONES for v in ("sageify", "sageify_grey")]
for seed in (0, 1, 2):
    files += [f"{BASE}/paper_recipe_keep/adjusted_seed{seed}_{end}" for end in (
        "summary.json", "history.csv", "sage_probs.csv", "sage_fov_probs.csv", "sage_masked_probs.csv",
        "geometry_summary.json", "geometry_sage_probs.csv", "geometry_hk_probs.csv")]
total = 0
for path in files:
    shown = path.replace(BASE, "MyDrive/sage-shift")
    if os.path.isfile(path):
        total += os.path.getsize(path)
        print(f"{os.path.getsize(path) / 1e6:9.2f} MB  {shown}")
    else:
        print(f"   MISSING    {shown}")
print(f"{len(files)} files, {total / 1e6:.0f} MB in total. Leave the *_best.pt checkpoints on Drive.")

In [ ]:
W = "/content/drive/MyDrive/sage-shift/weights/RN50_GastroNet-5M_DINOv1.pth"
PRE = "/content/drive/MyDrive/sage-shift/preprocess"
FEAT = "/content/drive/MyDrive/sage-shift/features"
!sha256sum {W}
!python scripts/06_extract_features.py --datasets sage hk --backbones resnet50_gastronet_dino --gastronet-weights {W} \
    --variants raw fov colour fov_colour --workers 2 --reference {PRE}/preprocess_reference.json --out-dir {FEAT}
!ls -l {FEAT}/sage/resnet50_gastronet_dino {FEAT}/hk/resnet50_gastronet_dino